## EDA on label from User

QUESTION 1: Find all possible unique values
- Category
- Subcategory
- Root_code
- Sub_code

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('label.xlsx')

print("--- 1. UNIQUE VALUES IN DATASET ---")
print(f"Categories ({df['Category'].nunique()}): {df['Category'].dropna().unique()}")
print(f"Subcategories ({df['Subcategory'].nunique()}): {df['Subcategory'].dropna().unique()}")
print(f"Root Codes ({df['root_code'].nunique()}): {df['root_code'].dropna().unique()}")
print(f"Sub Codes ({df['sub_code'].nunique()}): {df['sub_code'].dropna().unique()}\n")

QUESTION 2: Find mappings and human errors

In [ ]:
print("--- 2. MAPPING RULES & ANOMALY DETECTION ---")

# Group by the human text labels and see what codes they mapped to, counting occurrences
mapping_df = df.groupby(
    ['Category', 'Subcategory', 'root_code', 'sub_code'], 
    dropna=False
).size().reset_index(name='Count')

# Sort so that for each Category/Subcategory, the most frequent mapping is at the top
mapping_df = mapping_df.sort_values(
    by=['Category', 'Subcategory', 'Count'], 
    ascending=[True, True, False]
)

# Calculate the total times a Category/Subcategory pair appears
mapping_df['Total_Category_Occurrences'] = mapping_df.groupby(['Category', 'Subcategory'])['Count'].transform('sum')

# Calculate the percentage of times THIS specific mapping was used
mapping_df['Mapping_Confidence_%'] = (mapping_df['Count'] / mapping_df['Total_Category_Occurrences']) * 100

print("\nAll Mappings (Sorted by frequency):")
display(mapping_df) # Use print(mapping_df) if not in Jupyter

In [ ]:
# --- ISOLATING THE ERRORS ---
# Assuming that whatever mapping happens the majority of the time (>50%) is the "Correct" rule...
# Anything else is likely a human annotation error.

print("\n--- LIKELY HUMAN ANNOTATION ERRORS ---")
# Filter for mappings that happen less than 20% of the time for that specific category
anomalies = mapping_df[mapping_df['Mapping_Confidence_%'] < 20.0]

if anomalies.empty:
    print("Good news! No major anomalies found. The mapping is consistent.")
else:
    print("Found potential mislabels! Annotators likely messed these up:")
    display(anomalies)

In [ ]:
# ==========================================
# CELL 1: EYEBALL SPECIFIC CATEGORIES
# ==========================================
from IPython.display import Image, display
import os

# 1. Define the specific pair you want to investigate
TARGET_CATEGORY = "Insert Category Here"       # e.g., "Medical Document"
TARGET_SUBCAT = "Insert Subcategory Here"      # e.g., "Lab Results"

# 2. Filter the existing 'df' for this specific pair
subset_df = df[(df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)]

print(f"Found {len(subset_df)} documents for {TARGET_CATEGORY} -> {TARGET_SUBCAT}\n")

# 3. Display the images (Showing the first 10 to avoid crashing the notebook)
for idx, row in subset_df.head(10).iterrows():
    print("-" * 50)
    print(f"Row Index: {idx}")
    print(f"Current Labels -> root_code: [{row['root_code']}] | sub_code: [{row['sub_code']}]")
    print(f"Filepath: {row['Filepath']}")
    
    filepath = str(row['Filepath'])
    if os.path.exists(filepath):
        # Display the image directly in the notebook output
        display(Image(filename=filepath, width=600))
    else:
        print(f"Image not found at path: {filepath}")

In [ ]:
# ==========================================
# CELL 2: BATCH REPLACE & SAVE
# ==========================================

# 1. Define the pair you want to fix
TARGET_CATEGORY = "Insert Category Here"
TARGET_SUBCAT = "Insert Subcategory Here"

# 2. Define the CORRECT codes they should be mapped to
CORRECT_ROOT_CODE = "NON"
CORRECT_SUB_CODE = "OTH"

# 3. Create the filter condition
condition = (df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)

# 4. Use .loc to safely update the values in the original dataframe
df.loc[condition, 'root_code'] = CORRECT_ROOT_CODE
df.loc[condition, 'sub_code'] = CORRECT_SUB_CODE

print(f"Successfully updated {condition.sum()} rows to {CORRECT_ROOT_CODE} -> {CORRECT_SUB_CODE}.")

# 5. Export the cleaned data to CSV
output_filename = 'label_cleaned.csv'
df.to_csv(output_filename, index=False)
print(f"Cleaned dataset exported as: {output_filename}")

## Evaluation

In [ ]:
# ==========================================
# v3 CHANGELOG (vs v2)
# ==========================================
# 1. AGENT1 (triage): explicitly instructs the model to ignore the AIA insurance
#    watermark disclaimer as a routing signal -- it was causing real ID/financial docs
#    to be misrouted to trash_others.
# 2. AGENT1 (triage): added a "native structure vs eForm field" test so an ID number /
#    premium amount embedded inside an insurance application template no longer gets
#    mistaken for a real ID/financial document.
# 3. AGENT1 (triage): added OCR-noise-tolerance guidance for medical docs with garbled
#    but technical-looking OCR text (blurry scans), and a partial-legibility rule for
#    foreign-script (e.g. Lao) documents with mostly-unreadable OCR.
# 4. AGENT1 + AGENT2: both now told the OCR may be markdown with layout tags, and to use
#    table/header structure as an anchor.
# 5. AGENT2 (specialist) + schema: merged id_fullthaivisa + id_passportstamp into a single
#    id_visastamp leaf (they almost always co-occur on one page), with anchor guidance for
#    "DEPARTED"/"ADMITTED"/"ENTRY" keywords and garbled curved-stamp digits.
# 6. AGENT2 (specialist): added a defense-in-depth check to catch insurance eForms that
#    leak past triage, routing them to financial_others/id_others instead of forcing a
#    specific wrong subtype.
# AGENT3 (medical specialist) intentionally left untouched -- owned by a different prompt owner.
# ==========================================

import math
import pandas as pd
from typing import Literal, Optional
from pydantic import BaseModel
import csv
import time
import random
import threading
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import AzureOpenAI, BadRequestError, RateLimitError, APIConnectionError, APITimeoutError
try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

# ==========================================
# 1. DEFINE STRUCTURED OUTPUT SCHEMAS
# ==========================================

class TriageOutput(BaseModel):
    chain_of_thought: str
    routing_decision: Literal["medical", "processable_non_medical", "trash_others"]

class NonMedSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "financial_bankstatement", "financial_bookbank",
        "financial_companyregistration", "financial_others",
        "id_driverlicense", "id_fatca_w9", "id_foreignerconfirmationform",
        "id_foreigner_nationalid", "id_visastamp", "id_passport",
        "id_statelessid", "id_thainationalid",
        "id_workpermit", "id_others"
        # NOTE: id_fullthaivisa + id_passportstamp merged into id_visastamp -- these two
        # almost always appear on the same physical page, so splitting them was creating
        # noisy boundary cases with no real downstream benefit.
    ]

class MedSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "medical_clinical", "medical_healthcheck",
        "medical_lab", "medical_others"
    ]

# ==========================================
# 2. DEFINE PROMPT VARIABLES
# ==========================================

# --- AGENT 1: TRIAGE PROMPTS ---
# CHANGED (v4): same decision logic as v3, rewritten for token efficiency -- collapsed
# repeated warnings, pulled the eForm test out as one shared rule instead of restating it
# under both 'processable_non_medical' and 'trash_others', shorter sentences throughout.
# ~35% fewer tokens than v3 with no loss of the watermark / eForm / OCR-noise / partial-
# legibility fixes.
AGENT1_SYS_PROMPT = """You are a strict Triage Routing Agent. Classify raw OCR/markdown text into exactly one category: `medical`, `processable_non_medical`, or `trash_others`. Treat the anchors below as supporting evidence for a pattern, not a checklist of exact strings -- read holistically. Ignore PII.
 
### Global Rules
- **Ignore Watermarks:** Insurance disclaimers (+ date/time stamps) are never evidence of document type -- ignore them.
- **OCR Artifact Override:** If text is mostly garbled but contains any of: "VISA", "VISA CLASS", "VISACLASS", "IMMIGRATION", "DEPARTED", "ADMITTED", "ENTRY" (standardized terms, safe to match literally) -> route to `processable_non_medical`, even if your overall read is "mostly noise." If you notice one of these and still lean trash_others, that's the signal to route processable instead.
- **Administrative Metadata Is Not Content:** Postal/mailing/envelope text describes a document without containing it (even if it names what's inside, e.g. "sending a health check report") -> `trash_others`.
 
---
### 1. medical
Requires the **clinical content itself** (notes, lab metrics, health-check parameters, prescriptions, test ranges, normal/abnormal flags) -- not a reference to it.
- **INCLUDE:** Garbled OCR with technical medical terms next to numbers/units/ranges (flag low confidence in chain_of_thought).
- **EXCLUDE -> trash_others:** Billing/invoices/receipts. Any letter/memo/eForm that discusses, requests, or forwards medical info without the clinical data itself -- even hospital-stamped, even with a signature block. Ask: is this the record, or correspondence *about* one?
 
### 2. processable_non_medical
Requires **native document structure**, not fields filled into an application wrapper. Route here if ANY pattern matches, even fragmented/illegible:
- **ID/Identity:** Issuing country name/code (e.g. "LAO PDR", "RDP LAO"), a genuine name+DOB or name+issue/expiry block, "รับรองสำเนาถูกต้อง" (certified true copy), or a `<figure>` region actually wrapping such a personal-detail block -- not just any name/reference number sitting near a `<figure>` tag with no other identity structure.
- **Foreign/Stateless ID:** Stateless-ID text, or garbled non-Thai script (e.g. Lao misread as Thai) combined with a country-code header.
- **Passports/Visas:** MRZ lines (letters/digits + long "<" runs, e.g. "PA123SD464987<<<<<<<"), or the immigration keywords above near messy digits/dates.
- **Financial/Corporate:** salary/income disclosure (เงินเดือน), tax-withholding declarations (W-9/FATCA-style), corporate registration + certification signatures, bank account + branch, or a native transaction ledger (dated money movements, ideally a markdown table). A document counts here if it has independent substance beyond an insurance-application wrapper -- see the litmus test below.
 
### 3. trash_others (Default & eForm Trap)
Route here if no pattern above matches, or the document fails this test: **strip out the applicant's name, ID number, and policy number -- is any standalone financial, clinical, or identity statement left?** If all that remains is "applying for / updating / requesting details about an insurance policy," it's an eForm wrapper -> trash, regardless of medical or financial-sounding language elsewhere.
- **eForms/Applications/Proposals:** IDs, DOBs, or premiums that are just filled-in fields, including proposal tables of premium/sum-insured totals (not bank ledgers).
- **Memos/Bare Codes:** Clarification notes with nothing independent per the litmus test, or a bare name + single policy-style code with no other structure.
- **Photos:** Portraits/ID placeholders with no issuing text.
- **Medical Admin:** Billing, receipts, non-clinical cover letters, delivery/postal metadata (see Global Rules).
 
---
### Output Format
In `chain_of_thought`: state the pattern found (or "none"), and note if you're overriding a "looks like noise" impression, applying the eForm litmus test, or disregarding the insurance watermark."""
 
AGENT1_USER_PROMPT = "Evaluate and triage the following raw OCR text:\n{ocr_text}"

# --- AGENT 2: NON-MED SPECIALIST PROMPTS ---
AGENT2_SYS_PROMPT = """You are a strict Non-Medical Document Specialist. Categorize pre-verified Financial or Identification OCR text into the exact subclass.

### Global Rules
- **Ignore Watermarks:** Disregard insurance disclaimers (+ date/time stamps).
- **Rely on Structure:** Use markdown tags (<table>, <figure>) and specific keywords. Do not process PII.
- **Garbled Text is Expected:** Non-Thai IDs (e.g., Lao) often yield strange/garbled Thai text. Rely on legible anchors (country codes, MRZ, English keywords).

### Financial Subclasses
- **financial_bankstatement:** Running transactional ledgers, money transfers, balances. Expect dated transaction rows inside a `<table>`.
- **financial_bookbank:** Static account identity headers (bilingual accounts, branches, bank names). Lacks long transaction ledgers.
- **financial_companyregistration:** Commercial corporate registry verbiage, certification signatures.
- **financial_others:** Salary/payroll slips ("เงินเดือน") or generic financial documents not fitting the above.

### Identification Subclasses
- **id_thainationalid:** Thai demographic card headers. Often contains "รับรองสำเนาถูกต้อง" (certified true copy) or `<figure>` tags wrapping name/issue/expiry/DOB.
- **id_foreigner_nationalid:** Look for country headers like "LAO PDR" or "RDP LAO" mixed with strange/garbled OCR text and basic ID fields.
- **id_statelessid:** Look for "บัตรประจำตัวผู้คนที่ไม่มีสัญชาติไทย" or text indicating stateless/no-registered-nationality status.
- **id_passport:** Bio data page. Look for MRZ-style lines at the bottom (e.g., letters/digits with long runs of `<` like `PA123<<<<<<<`).
- **id_visastamp:** Immigration control stamps or Thai visas. Anchor keywords: "VISA", "VISACLASS", "IMMIGRATION", "DEPARTED", "ADMITTED", "ENTRY". Text is often curved/garbled.
- **id_workpermit:** Labor authorization rules, official employment permissions.
- **id_fatca_w9:** US tax withholding terminology, backup withholding rules, entity declarations.
- **id_foreignerconfirmationform:** Government-issued confirmation of foreign status (authority header, official stamp). NOT an insurance form.
- **id_others:** Torn, partial, or borderline official ID artifacts that do not cleanly match the specific ID subtypes above.

In `chain_of_thought`: State the structural anchors, keywords (e.g., "LAO PDR", "เงินเดือน"), or layout tags found before selecting the subclass."""

AGENT2_USER_PROMPT = "Perform deep classification on this verified processable text:\n{ocr_text}"

# --- AGENT 3: MEDICAL SPECIALIST PROMPTS ---
AGENT3_SYS_PROMPT = """You are a strict Medical Document Specialist. Categorize pre-verified medical OCR text into the exact clinical subclass. Focus on clinical structures and specific medical anchors. Do not extract PII.

### Subclass Priority & Evaluation Guides:

- **medical_lab (HIGHEST PRIORITY):** Laboratory test results, which almost always contain tabular data (look for markdown `<table>`, `<tr>`, `<td>`). Look for quantitative lab panels, blood/chemical analysis, measurement units (e.g., "mg/dL", "mmol/L"), and diagnostic indicators ("positive", "negative").
  - *CRITICAL RULE:* If specific lab test results are present, you MUST route here, EVEN IF the document also contains vital signs (BP, BMI, etc.) or clinical notes. 

- **medical_healthcheck (Secondary Priority):** Look for extractable vital signs and physical measurements: T. (Temperature), P. (Pulse), RR., BP. (Blood Pressure), นน. (Weight), สูง (Height), or BMI. 
  - *CRITICAL RULE:* If the document contains these vital values but NO specific lab panels, route it here. Also includes standard checkup checklists (e.g., "normal", "abnormal", "ปกติ").

- **medical_clinical (Lowest Priority):** General outpatient/inpatient doctor's notes (OPD/IPD), treatment plans, symptom descriptions, and medication dosages (e.g., "mg", "ครั้ง"). 
  - *EXCLUSION:* Do not route here if extractable vital signs or lab panels are present (route to healthcheck or lab instead).

- **medical_others:** Medical documents that do not structurally fit the above categories. 
  - *SPECIFIC INCLUSION:* Always route **Sleep Test** reports to this category.

In `chain_of_thought`: State the specific clinical anchors found and explicitly mention the priority logic (e.g., "Found lab table with mg/dL, overriding vitals", "Found BP/BMI but no labs", "Sleep test mention") before making your final selection."""

AGENT3_USER_PROMPT = "Perform deep clinical classification on this verified medical text:\n{ocr_text}"

# ==========================================
# 3. CORE COGNITIVE PIPELINE
# ==========================================

def calculate_confidence(response_logprobs) -> float:
    """Whole-response confidence (legacy). Kept for backward compatibility / comparison.
    NOTE: this dilutes the signal with free-text chain_of_thought tokens -- prefer
    calculate_field_confidence for anything you actually act on (e.g. review-queue thresholds)."""
    if not response_logprobs or not response_logprobs.content:
        return 0.0
    token_logprobs = [t.logprob for t in response_logprobs.content if t.logprob is not None]
    if not token_logprobs:
        return 0.0
    avg_logprob = sum(token_logprobs) / len(token_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def calculate_field_confidence(response_logprobs, field_value: str) -> float:
    """Isolates and averages logprobs only for the tokens making up the classification
    enum value itself (e.g. 'processable_non_medical' or 'id_passport'), rather than the
    whole JSON payload including the free-text chain_of_thought. This is a much better
    proxy for "how sure was the model about the label" and is what should feed a
    human-review threshold.

    Falls back to calculate_confidence if the token span can't be located (e.g. due to
    tokenizer quirks), so it always returns a usable number.
    """
    if not response_logprobs or not response_logprobs.content:
        return 0.0

    tokens = response_logprobs.content
    token_strs = [t.token for t in tokens]

    # Reconstruct the running string and find which token indices correspond to
    # the field_value substring (the enum's own text), searching from the end since
    # 'routing_decision'/'subcategory' is emitted after chain_of_thought in our schemas.
    running = ""
    span_start, span_end = None, None
    for i, tok in enumerate(token_strs):
        prev_len = len(running)
        running += tok
        if field_value in running[max(0, prev_len - len(field_value)):]:
            end_idx = i
            # walk backward to find the first token index where field_value starts
            partial = ""
            start_idx = i
            for j in range(i, -1, -1):
                partial = token_strs[j] + partial
                start_idx = j
                if field_value in partial:
                    break
            span_start, span_end = start_idx, end_idx

    if span_start is None:
        return calculate_confidence(response_logprobs)

    span_logprobs = [tokens[i].logprob for i in range(span_start, span_end + 1) if tokens[i].logprob is not None]
    if not span_logprobs:
        return calculate_confidence(response_logprobs)

    avg_logprob = sum(span_logprobs) / len(span_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)

def call_with_retry(fn, *args, max_retries: int = 6, base_delay: float = 1.0,
                     max_delay: float = 60.0, **kwargs):
    """Wraps a single API call with retry + exponential backoff for 429s and transient
    connection errors. Respects Azure's Retry-After header when present (this is almost
    always a better signal than guessing your own backoff timing)."""
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except RateLimitError as e:
            retry_after = None
            try:
                retry_after = float(e.response.headers.get("retry-after"))
            except Exception:
                pass
            delay = retry_after if retry_after is not None else min(max_delay, base_delay * (2 ** attempt))
            delay += random.uniform(0, delay * 0.25)  # jitter avoids thundering-herd retries
            print(f"[RATE LIMIT] attempt {attempt + 1}/{max_retries}, sleeping {delay:.1f}s")
            time.sleep(delay)
        except (APIConnectionError, APITimeoutError) as e:
            delay = min(max_delay, base_delay * (2 ** attempt)) + random.uniform(0, 1)
            print(f"[TRANSIENT ERROR] {type(e).__name__}: {e} -- sleeping {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Exceeded max_retries ({max_retries}) calling {getattr(fn, '__name__', fn)}")


def run_cascade_pipeline(ocr_text: str, client: AzureOpenAI, model_name: str = "gpt-4o") -> dict:
    """Executes the three-stage cascading architecture on a single piece of text."""

    # ----------------------------------------------------
    # STAGE 1: Triage Routing
    # ----------------------------------------------------
    triage_response = client.beta.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": AGENT1_SYS_PROMPT},
            {"role": "user", "content": AGENT1_USER_PROMPT.format(ocr_text=ocr_text)}
        ],
        response_format=TriageOutput,
        logprobs=True,
        top_logprobs=1
    )

    triage_data = triage_response.choices[0].message.parsed
    route = triage_data.routing_decision
    triage_conf = calculate_field_confidence(triage_response.choices[0].logprobs, route)

    # ----------------------------------------------------
    # STAGE 2A: Conditional Non-Med Processing
    # ----------------------------------------------------
    if route == "processable_non_medical":
        spec_response = client.beta.chat.completions.parse(
            model=model_name,
            messages=[
                {"role": "system", "content": AGENT2_SYS_PROMPT},
                {"role": "user", "content": AGENT2_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=NonMedSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)

        return {
            "triage_decision": route,
            "triage_reason": triage_data.chain_of_thought,
            "triage_confidence": triage_conf,
            "final_subcategory": spec_data.subcategory,
            "specialist_reason": spec_data.chain_of_thought,
            "specialist_confidence": spec_conf
        }

    # ----------------------------------------------------
    # STAGE 2B: Conditional Med Processing
    # ----------------------------------------------------
    elif route == "medical":
        spec_response = client.beta.chat.completions.parse(
            model=model_name,
            messages=[
                {"role": "system", "content": AGENT3_SYS_PROMPT},
                {"role": "user", "content": AGENT3_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=MedSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)

        return {
            "triage_decision": route,
            "triage_reason": triage_data.chain_of_thought,
            "triage_confidence": triage_conf,
            "final_subcategory": spec_data.subcategory,
            "specialist_reason": spec_data.chain_of_thought,
            "specialist_confidence": spec_conf
        }

    # If routed to Trash, exit early without calling any specialist agent
    return {
        "triage_decision": route,
        "triage_reason": triage_data.chain_of_thought,
        "triage_confidence": triage_conf,
        "final_subcategory": f"routed_to_{route}",
        "specialist_reason": "Skipped - Cleaned out as unstructured noise by triage agent.",
        "specialist_confidence": 100.0
    }

In [ ]:
# ==========================================
# 4. DATAFRAME EVALUATION EXECUTION
# ==========================================
 
_checkpoint_lock = threading.Lock()
 
def _process_one_row(idx, row, client, triage_model, nonmed_model, med_model):
    """Runs the cascade pipeline for a single row, with the same error handling as
    before (content filter block vs. any other exception), just factored out so it can
    be submitted to a thread pool."""
    ocr_text = str(row['ocr_text'])
 
    try:
        res = run_cascade_pipeline(ocr_text, client, triage_model, nonmed_model, med_model)
    except BadRequestError as e:
        error_message = str(e).lower()
        if "content management policy" in error_message or "jailbreak" in error_message:
            print(f"[WARNING] Doc {idx + 1} blocked by Azure Jailbreak Filter. Skipping...")
            res = {
                "triage_decision": "blocked_by_firewall",
                "triage_reason": "Azure OpenAI Content Filter flagged this OCR text as a jailbreak attempt.",
                "triage_confidence": 0.0,
                "final_subcategory": "error_content_filter",
                "specialist_reason": "Skipped due to API security policy.",
                "specialist_confidence": 0.0
            }
        else:
            raise e
    except Exception as e:
        print(f"[ERROR] Doc {idx + 1} failed due to unexpected error: {e}")
        res = {
            "triage_decision": "api_error",
            "triage_reason": str(e),
            "triage_confidence": 0.0,
            "final_subcategory": "api_error",
            "specialist_reason": "API disconnected or failed.",
            "specialist_confidence": 0.0
        }
 
    combined_row = {
        "filename": row.get('filename'),
        "input_category_label": row.get('category'),
        "input_subcategory_label": row.get('subcategory'),
        **res
    }
    return idx, combined_row

In [ ]:
def evaluate_experiment(csv_path: str, endpoint: str, api_key: str,
                         triage_model: str = "gpt-5-mini",
                         nonmed_model: str = "gpt-5",
                         med_model: str = "gpt-5",
                         max_workers: int = 5,
                         checkpoint_path: str = "experiment_full_cascade_results.csv",
                         resume: bool = True):
    """Reads the evaluation data, executes the cascade pipeline concurrently, and
    returns results.
 
    triage_model / nonmed_model / med_model: deployment names for each stage. Defaults
        reflect a cost/performance split -- cheaper model where accuracy is already good
        (triage), stronger model where accuracy is the actual bottleneck (specialists).
        These must match deployment names that exist in your Azure resource, not the
        raw OpenAI model family names -- rename to whatever you deployed them as.
    max_workers: number of documents processed in parallel. Start conservative (3-5) and
        raise it while watching for [RATE LIMIT] log lines -- if you see frequent 429s
        even after backoff, max_workers is higher than your Azure deployment's TPM/RPM
        quota supports. Check your quota under Azure OpenAI Studio > Quotas.
    checkpoint_path: results are appended here as each document finishes, so a crash
        partway through doesn't lose completed work.
    resume: if True and checkpoint_path already exists, filenames already present there
        are skipped instead of being re-processed.
    """
 
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2024-08-01-preview"
    )
 
    df = pd.read_csv(csv_path)
 
    already_done = set()
    if resume and os.path.isfile(checkpoint_path):
        prior = pd.read_csv(checkpoint_path)
        if 'filename' in prior.columns:
            already_done = set(prior['filename'].dropna().astype(str))
        print(f"Resuming: {len(already_done)} document(s) already in checkpoint, will be skipped.")
 
    pending = [
        (idx, row) for idx, row in df.iterrows()
        if str(row.get('filename')) not in already_done
    ]
 
    print(f"Starting cascade experiment on {len(pending)} document(s) "
          f"(of {len(df)} total, {len(already_done)} already done) with max_workers={max_workers}...")
 
    fieldnames = None  # inferred from the first completed row
 
    def write_row(combined_row):
        nonlocal fieldnames
        with _checkpoint_lock:
            file_exists = os.path.isfile(checkpoint_path)
            if fieldnames is None:
                if file_exists:
                    fieldnames = list(pd.read_csv(checkpoint_path, nrows=0).columns)
                else:
                    fieldnames = list(combined_row.keys())
            with open(checkpoint_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()
                writer.writerow(combined_row)
 
    results_by_idx = {}
    completed = 0
    iterator_kwargs = dict(total=len(pending)) if HAS_TQDM else {}
 
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_process_one_row, idx, row, client, triage_model, nonmed_model, med_model): idx
            for idx, row in pending
        }
 
        progress = tqdm(as_completed(futures), **iterator_kwargs) if HAS_TQDM else as_completed(futures)
 
        for future in progress:
            idx, combined_row = future.result()
            write_row(combined_row)
            results_by_idx[idx] = combined_row
            completed += 1
            if not HAS_TQDM:
                print(f"[{completed}/{len(pending)}] {combined_row.get('filename')} "
                      f"Triage: {combined_row['triage_decision']} -> Leaf: {combined_row['final_subcategory']}")
 
    print(f"\nDone. {completed} document(s) processed this run. "
          f"Full results (including any resumed rows) are in '{checkpoint_path}'.")
 
    output_df = pd.read_csv(checkpoint_path)
    return output_df
 
# To run in your notebook cell:
# df_results = evaluate_experiment(
#     "test_data.csv",
#     "https://your-endpoint.openai.azure.com/",
#     "your-api-key",
#     triage_model="gpt-5-mini",   # deployment name for the cheap/already-accurate stage
#     nonmed_model="gpt-5",        # deployment name for the accuracy bottleneck stage
#     med_model="gpt-5",           # swap to whatever your senior wants for medical
#     max_workers=5,                # tune to your Azure quota
# )